# Design Decisions

## Initial Rating
- Every new team starts at 1500.

## Unknown Teams
- Automatically initialize new teams.

## Training Period
- 1 January 2000 onwards.

## Training Matches
- Only completed matches.

In [1]:
ratings = {}

def initialize_team(team):
    if team not in ratings:
        ratings[team] = 1500

# Day 6 - Building the Elo Rating System

## Goal

Today we begin implementing our own Elo rating engine.

Objectives:

- Understand expected probability
- Implement the Elo expected score formula
- Test the formula on sample teams

In [3]:
import pandas as pd
import numpy as np

In [5]:
matches = pd.read_csv(
    r"E:\Python\worldcup-intelligence-platform\data\raw\clean_matches.csv",
    parse_dates=["date"]
)

In [6]:
initialize_team("Brazil")
initialize_team("Argentina")
initialize_team("Brazil")

ratings

{'Brazil': 1500, 'Argentina': 1500}

In [9]:
def expected_score(home_team_rating, away_team_rating):
    expected = 1 / (1 + 10 ** ((away_team_rating - home_team_rating) / 400))
    return expected

In [10]:
print(expected_score(1500, 1500))

0.5


In [11]:
print(expected_score(1700, 1500))

0.7597469266479578


In [12]:
print(expected_score(1500, 1700))

0.2402530733520421


In [13]:
print(expected_score(2000, 1500))

0.9467597847979775


In [17]:
def actual_score(home_score, away_score):
    if home_score > away_score:
        return 1, 0

    elif away_score > home_score:
        return 0, 1

    else:
        return 0.5, 0.5

In [18]:
print(actual_score(3, 0))

(1, 0)


In [19]:
print(actual_score(1, 1))

(0.5, 0.5)


In [20]:
print(actual_score(0, 2))

(0, 1)


In [22]:
def update_elo(team_rating, expected, actual, k=40):
    new_rating = team_rating + k*(actual - expected)
    return new_rating

In [30]:
ratings = {}

for _, row in matches.iterrows():

    # Initialize teams
    initialize_team(row["home_team"])
    initialize_team(row["away_team"])

    # Current ratings
    home_rating = ratings[row["home_team"]]
    away_rating = ratings[row["away_team"]]

    # Expected scores
    home_expected = expected_score(home_rating, away_rating)
    away_expected = expected_score(away_rating, home_rating)

    # Actual scores
    home_actual, away_actual = actual_score(
        row["home_score"],
        row["away_score"]
    )

    # Updated ratings
    home_new = update_elo(
        home_rating,
        home_expected,
        home_actual
    )

    away_new = update_elo(
        away_rating,
        away_expected,
        away_actual
    )

    # Save back
    ratings[row["home_team"]] = home_new
    ratings[row["away_team"]] = away_new

In [31]:
len(ratings)

321

In [32]:
ratings

{'Egypt': 1786.9425255802373,
 'Togo': 1501.6550171695524,
 'Tunisia': 1648.9970157363712,
 'Trinidad and Tobago': 1496.2700641859356,
 'Canada': 1807.3134713132617,
 'Mexico': 1929.155593448191,
 'Iran': 1823.5224747838347,
 'Ivory Coast': 1813.3778182843814,
 'Burkina Faso': 1620.445800539942,
 'Gabon': 1516.092566872766,
 'Guatemala': 1581.8922041743272,
 'Armenia': 1411.092986111226,
 'Bermuda': 1387.5324562905273,
 'Cameroon': 1672.1087988446427,
 'Senegal': 1799.0735009373832,
 'China': 1558.1319578211953,
 'New Zealand': 1618.1122100232737,
 'Jamaica': 1611.7805799925097,
 'United States': 1836.7178039292787,
 'Morocco': 1951.8444696541005,
 'Ghana': 1608.9326194942707,
 'Panama': 1691.0518948081915,
 'Algeria': 1853.5388713885447,
 'Malta': 1401.1964074917869,
 'Qatar': 1550.8898945334622,
 'South Korea': 1797.8703154144432,
 'Philippines': 1405.7066117695674,
 'Zambia': 1509.062415656039,
 'Nigeria': 1782.6517277251855,
 'South Africa': 1673.1261380831224,
 'Vietnam': 1550.971

In [33]:
list(ratings.items())[:10]

[('Egypt', 1786.9425255802373),
 ('Togo', 1501.6550171695524),
 ('Tunisia', 1648.9970157363712),
 ('Trinidad and Tobago', 1496.2700641859356),
 ('Canada', 1807.3134713132617),
 ('Mexico', 1929.155593448191),
 ('Iran', 1823.5224747838347),
 ('Ivory Coast', 1813.3778182843814),
 ('Burkina Faso', 1620.445800539942),
 ('Gabon', 1516.092566872766)]

In [34]:
sorted_ratings = sorted(ratings.items(), key=lambda x: x[1], reverse=True)

In [35]:
sorted_ratings[:20]

[('Argentina', 2069.158450737571),
 ('Spain', 2036.5631271370216),
 ('France', 1997.5820785250396),
 ('Brazil', 1956.8830595366148),
 ('Morocco', 1951.8444696541005),
 ('Portugal', 1950.0943002845952),
 ('Colombia', 1949.037441552032),
 ('England', 1935.939150559108),
 ('Germany', 1934.3105568667966),
 ('Mexico', 1929.155593448191),
 ('Netherlands', 1927.0887868458954),
 ('Japan', 1926.316242371596),
 ('Ecuador', 1891.722591386223),
 ('Norway', 1880.6134288570563),
 ('Italy', 1874.948880137349),
 ('Croatia', 1870.4609259417539),
 ('Switzerland', 1861.7740239731268),
 ('Belgium', 1858.2137091000445),
 ('Algeria', 1853.5388713885447),
 ('Turkey', 1853.3147688327106)]

# Baseline Elo Model (Version 1)

## Assumptions

- Every team starts at 1500.
- K-factor = 40 for every match.
- Goal difference is ignored.
- Every tournament has equal importance.
- Matches are processed chronologically.
- Elo ratings are updated after every match.

## Limitations

- Friendlies count the same as World Cup Finals.
- Winning 1–0 and 8–0 have the same effect.
- Teams do not have historical ratings before the year 2000.

In [36]:
matches = pd.read_csv(
    "../data/processed/clean_matches.csv",
    parse_dates=["date"]
)